In [1]:
import numpy as np
import pandas as pd

import tubesml as tml
from tubesml.base import BaseTransformer, fit_wrapper, transform_wrapper

import matplotlib.pyplot as plt
%matplotlib inline

/home/lucabasa/Git/kaggle_competitions/nfl_2026/.nfl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [30]:
combine_results = pd.read_csv("data/combine_results.csv")

career_success = pd.read_csv("data/player_career_successes.csv")

career_success["per_starting"] = career_success["career_games_started"] / career_success["career_games_active"]
career_success["total_career_snaps"] = career_success[[c for c in career_success if "snaps" in c]].sum(axis=1)
career_success["offensive_snaps_pg"] = career_success["career_offensive_snaps"] / career_success["career_games_active"]
career_success["defensive_snaps_pg"] = career_success["career_defensive_snaps"] / career_success["career_games_active"]
career_success["special_snaps_pg"] = career_success["career_special_teams_snaps"] / career_success["career_games_active"]
career_success["total_snaps_pg"] = career_success["total_career_snaps"] / career_success["career_games_active"]

career_success["tot_awards"] = career_success[[c for c in career_success if "pro_" in c]].sum(axis=1)

players = pd.read_csv("data/players.csv")

players_play = pd.read_csv("data/player_play.csv")

games = pd.read_csv("data/games.csv")

plays = pd.merge(players_play, games, on="game_id")
plays = pd.merge(plays, players[["nfl_id", "draft_year"]], on="nfl_id")

plays["season_n"] = plays["season"] - plays["draft_year"] + 1

plays["is_home"] = (plays["team_abbr"] == plays["home_team_abbr"]).astype(bool)

stats = ['pass_length', 'yards_gained', 'rec_yards',
       'rush_yards', 'yards_after_catch', 'fumble', 'fumble_lost', 'sack', 'tackle', 'assist', 'caused_forced_fumble',
       'tackle_for_loss']

plays_counts = plays.groupby(["nfl_id", "season_n", "season_type"])[stats].sum().astype(int).reset_index()

In [31]:
df = players.copy()
df = pd.merge(df, combine_results, on=["nfl_id", "draft_year"])
df = pd.merge(df, career_success, on="nfl_id")
#df = pd.merge(df, plays_counts)

df.head()

,nfl_id,display_name,draft_year,nfl_position,birth_date,college_name,college_conference,draft_round,draft_pick_within_round,draft_overall_pick,...,ap_all_pro_1st_team,ap_all_pro_2nd_team,pro_bowl_original_ballot,per_starting,total_career_snaps,offensive_snaps_pg,defensive_snaps_pg,special_snaps_pg,total_snaps_pg,tot_awards
0,55867,Will Anderson,2023,DE,2001-09-02,Alabama,Southeastern Conference,1.0,3.0,3.0,...,1,0,1,0.862745,1976,0.000000,36.941176,1.803922,38.745098,2
1,55870,Paris Johnson,2023,T,2001-07-31,Ohio State,Big Ten Conference,1.0,6.0,6.0,...,0,0,0,0.914894,2971,59.617021,0.000000,3.595745,63.212766,0
2,55874,Darnell Wright,2023,T,2001-08-10,Tennessee,Southeastern Conference,1.0,10.0,10.0,...,0,1,0,0.960784,3389,63.176471,0.000000,3.274510,66.450980,1
3,55875,Peter Skoronski,2023,G,2001-07-31,Northwestern,Big Ten Conference,1.0,11.0,11.0,...,0,0,0,0.941176,3181,59.039216,0.000000,3.333333,62.372549,0
4,55877,Lukas Van Ness,2023,DE,2001-07-06,Iowa,Big Ten Conference,1.0,13.0,13.0,...,0,0,0,0.039216,1443,0.000000,20.705882,7.588235,28.294118,0


- add if the play is about visitor or home
- add season number for player
- add counts per season number per player per season_type
- probability win changes


In [27]:
def process_plays_data(plays):
    season_types = plays["season_type"].unique()
    
    # Group by nfl_id, season_n, and season_type to get counts
    plays_counts = plays.groupby(["nfl_id", "season_n", "season_type"])[stats].sum().astype(int).reset_index()
    
    # Get all unique nfl_id and season_n combinations
    unique_combinations = plays_counts[['nfl_id', 'season_n']].drop_duplicates()
    
    # Create complete season type combinations for each nfl_id/season_n pair
    complete_data = []
    for _, combo in unique_combinations.iterrows():
        nfl_id = combo['nfl_id']
        season_n = combo['season_n']
        
        # For each combination, ensure all 3 season types are present
        for season_type in season_types:
            complete_data.append({
                'nfl_id': nfl_id,
                'season_n': season_n,
                'season_type': season_type
            })
    
    # Create DataFrame with all combinations
    complete_df = pd.DataFrame(complete_data)
    
    # Merge with original data to fill missing season types
    result = complete_df.merge(plays_counts, on=["nfl_id", "season_n", "season_type"], how="left")
    
    # Fill NaN values with 0s for all stats columns
    result[stats] = result[stats].fillna(0).astype(int)
    
    
    return result



(2766, 16)